# Exploração da API de deputados

Este notebook compara a lista atual de deputados com a lista filtrada pela legislatura 57 e conta os deputados por UF. Os resultados ficam apenas na memória do notebook.

In [1]:
from time import sleep
from typing import Any

import pandas as pd
import requests

API_URL = "https://dadosabertos.camara.leg.br/api/v2/deputados"
REQUEST_TIMEOUT = 30
PAGE_SIZE = 100
REQUEST_DELAY_SECONDS = 0.2
MAX_ATTEMPTS = 3

## Funções para consultar todas as páginas

A primeira página usa os parâmetros da consulta. As páginas seguintes usam o URL `next` informado pela própria API. Cada tentativa tem timeout e no máximo três tentativas; entre páginas há uma pequena pausa para respeitar os limites da API.

In [2]:
def get_json(url: str, params: dict[str, int] | None = None) -> dict[str, Any]:
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            if attempt == MAX_ATTEMPTS:
                raise
            sleep(REQUEST_DELAY_SECONDS * attempt)
    raise RuntimeError("Request attempts finished unexpectedly.")


def fetch_deputies(legislature_id: int | None = None) -> list[dict[str, Any]]:
    params = {"itens": PAGE_SIZE, "ordem": "ASC", "ordenarPor": "nome"}
    if legislature_id is not None:
        params["idLegislatura"] = legislature_id

    deputies: list[dict[str, Any]] = []
    next_url: str | None = API_URL
    first_page = True

    while next_url is not None:
        page = get_json(next_url, params=params if first_page else None)
        deputies.extend(page["dados"])
        next_links = [link["href"] for link in page.get("links", []) if link.get("rel") == "next"]
        next_url = next_links[0] if next_links else None
        first_page = False
        if next_url is not None:
            sleep(REQUEST_DELAY_SECONDS)

    return deputies

## Buscar as duas listas

A primeira consulta não envia `idLegislatura`; a segunda envia `idLegislatura=57`.

In [3]:
deputies_all = fetch_deputies()
deputies_legislature_57 = fetch_deputies(legislature_id=57)

print(f"Total without legislature filter: {len(deputies_all)}")
print(f"Total in legislature 57: {len(deputies_legislature_57)}")

Total without legislature filter: 513
Total in legislature 57: 877


## Comparar ids

As diferenças abaixo mostram quais ids existem somente em cada resultado.

In [4]:
ids_all = {deputy["id"] for deputy in deputies_all}
ids_legislature_57 = {deputy["id"] for deputy in deputies_legislature_57}

only_without_filter = sorted(ids_all - ids_legislature_57)
only_in_legislature_57 = sorted(ids_legislature_57 - ids_all)

print("Ids only in the unfiltered list:", only_without_filter)
print("Ids only in legislature 57:", only_in_legislature_57)

Ids only in the unfiltered list: []
Ids only in legislature 57: [68720, 73531, 73806, 74052, 74065, 74095, 74140, 74141, 74159, 74283, 74295, 74299, 74696, 77003, 89536, 92346, 100689, 103758, 115200, 116379, 122195, 128760, 129618, 132504, 138286, 141480, 143942, 154700, 160531, 163321, 164360, 171139, 178909, 182146, 194260, 204376, 204422, 204433, 204448, 204476, 204503, 204507, 204514, 204522, 204558, 204573, 205865, 208297, 212625, 213274, 217480, 218086, 219585, 220008, 220531, 220575, 220577, 220579, 220580, 220587, 220593, 220596, 220600, 220601, 220619, 220627, 220634, 220636, 220639, 220659, 220705, 221339, 221378, 222142, 222429, 222565, 223128, 223398, 225386, 225727, 225730, 225846, 226179, 226410, 226553, 226708, 227275, 227310, 227323, 227324, 227370, 227401, 227433, 227660, 227800, 228042, 228272, 228616, 228797, 228837, 228941, 229069, 229082, 229106, 229112, 229225, 229259, 229333, 229432, 229585, 229939, 230088, 230177, 230762, 230764, 231911, 233218, 233598, 234253,

## Deputados por UF

A contagem usa a lista sem filtro de legislatura e o campo bruto `siglaUf` retornado pela API.

In [5]:
deputies_df = pd.DataFrame(deputies_all)
deputies_by_state = (
    deputies_df.groupby("siglaUf", dropna=False)["id"]
    .nunique()
    .sort_values(ascending=False)
    .rename("deputy_count")
    .reset_index()
)
deputies_by_state

,siglaUf,deputy_count
0,SP,70
1,MG,53
2,RJ,46
3,BA,39
4,RS,31
5,PR,30
6,PE,25
7,CE,22
8,MA,18
9,PA,17
